# Occlusion JEPA — visualisations PCA 3D

Charge un checkpoint (`best.pt`) et explore la structure de l'espace latent :

1. **Nuage global** — tous les embeddings projetés en PCA 3D, colorés par la position **x** du disque ; les frames occluses sont des losanges colorés par la position de la barre.
2. **Trajectoires d'épisodes** — trajectoire latente réelle (bleu, noir = occlus) vs rollout prédit (orange) sur des épisodes qui traversent la barre.

**Comment lire** : représentation interprétable = le nuage s'organise en nappe continue suivant (x, y) ; interpolation réussie = la trajectoire orange suit la bleue *à travers* la zone occluse sans discontinuité, et en ressort au bon endroit.

Fonctionne en local (kernel `jepa-wms`, code du repo) comme sur Colab. CPU suffisant.

In [1]:
import os, sys

if os.path.exists("src"):  # exécution locale, depuis la racine du repo
    sys.path.insert(0, "src")
else:                      # Colab
    REPO_URL = "https://github.com/<TON_USER>/occlusion-jepa.git"  # ← à remplacer
    if not os.path.exists("occlusion-jepa"):
        !git clone {REPO_URL} occlusion-jepa
    %pip install -q -r occlusion-jepa/requirements.txt
    sys.path.insert(0, "occlusion-jepa/src")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Chargement du checkpoint

In [2]:
from occlusion_jepa.train import load_checkpoint

CKPT_PATH = "runs/2026-07-16_barre-mobile/best.pt"  # ← adapter au run voulu

cfg, encoder, ema_encoder, predictor, history = load_checkpoint(CKPT_PATH, device=device)
print(f"checkpoint : {CKPT_PATH}")
print(f"epochs entraînées : {len(history)}, dernière val_inv : {history[-1]['val_inv']:.4f}")

checkpoint : runs/2026-07-16_barre-mobile/best.pt
epochs entraînées : 57, dernière val_inv : 0.3636


/Users/julesvidegrain/miniconda3/envs/jepa-wms/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Nuage global + un épisode

Figures plotly interactives (rotation à la souris). Si rien ne s'affiche en local : `fig_cloud.show(renderer="browser")`.

In [3]:
from occlusion_jepa.viz import pca3d_figures

fig_cloud, fig_ep = pca3d_figures(cfg, encoder, predictor, device, n_seqs=300)
fig_cloud.show()
fig_ep.show()

## D'autres épisodes traversants

Chaque seed tire un nouveau jeu d'épisodes (et refit la PCA) — utile pour vérifier que l'interpolation à travers l'occlusion n'est pas un coup de chance d'un seul épisode.

In [4]:
for seed in [111, 222, 333]:
    _, fig_ep = pca3d_figures(cfg, encoder, predictor, device, n_seqs=150, seed=seed)
    fig_ep.show()